# 02 — Histogram HSV12 + SVM

Ảnh RGB224 được chuyển sang HSV. Histogram kết hợp ba kênh có 12 bin trên mỗi trục, tạo vector 1.728 chiều. Mỗi pixel đóng góp vào đúng một ô H–S–V; vị trí và hình dạng chữ số không được giữ trong vector.


In [ ]:
from pathlib import Path
import os, sys
# Bỏ comment nếu Kaggle gắn nhiều bộ code hoặc dữ liệu.
# os.environ['CS231_PROJECT_ROOT'] = '/kaggle/input/TEN-DATASET-CODE'
# os.environ['CS231_DATA_ROOT'] = '/kaggle/input/TEN-DATASET-ANH/DATA'
# os.environ['CS231_SVM_PREVIOUS_RESULTS'] = '/kaggle/input/.../CS231_SVM/results_224'
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cs231-matplotlib')
roots = [Path.cwd(), *Path.cwd().parents]
roots += [p for pattern in ('*', 'datasets/*/*') for p in Path('/kaggle/input').glob(pattern)]
if os.environ.get('CS231_PROJECT_ROOT'):
    roots = [Path(os.environ['CS231_PROJECT_ROOT'])]
projects = sorted({p.resolve() for p in roots if (p/'src/preprocessing.py').is_file()})
if len(projects) != 1:
    raise FileNotFoundError('Đặt CS231_PROJECT_ROOT trỏ tới thư mục chứa src.')
PROJECT_ROOT = projects[0]
sys.path.insert(0, str(PROJECT_ROOT))
from src.experiments.runtime import setup, restore_previous_output
DATA_ROOT, RESULTS_ROOT = setup('SVM', PROJECT_ROOT)

In [ ]:
restore_previous_output(RESULTS_ROOT, 'SVM', 'inputs/experiment_config.json')

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.SVM.pipeline import (
    CLASSES, FEATURE_FOLDERS, manifest_for_split, extract_matrix, l2,
    save_features, load_features, same_manifest, run_dev_grid, save_dev_result,
    save_json, hsv12, hog_raw,
)
from src.preprocessing import read_rgb224

## 1. Đọc manifest của bước chuẩn bị


In [ ]:
train_manifest = pd.read_csv(RESULTS_ROOT / 'inputs/train_manifest.csv')
dev_manifest = pd.read_csv(RESULTS_ROOT / 'inputs/dev_manifest.csv')
same_manifest(train_manifest, manifest_for_split(DATA_ROOT, 'train'))
same_manifest(dev_manifest, manifest_for_split(DATA_ROOT, 'dev'))
OUTPUT = RESULTS_ROOT / 'hsv12'
FEATURE_ID = 'HSV12'
FEATURE_PIPELINE = 'RGB224 -> joint HSV histogram 12x12x12 -> L2'

## 2. Minh họa histogram

Ba đồ thị chỉ là phép chiếu biên của histogram 3D sau L2. SVC nhận toàn bộ vector 1.728 chiều, không nhận ba đồ thị riêng.


In [ ]:
sample = read_rgb224(DATA_ROOT / train_manifest.relative_path.iloc[0])
vector = hsv12(sample)
cube = vector.reshape(12, 12, 12)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(sample)
axes[0].axis('off')
for ax, marginal, title in zip(axes[1:],
    [cube.sum(axis=(1, 2)), cube.sum(axis=(0, 2)), cube.sum(axis=(0, 1))],
    ['Hue', 'Saturation', 'Value']):
    ax.bar(range(12), marginal)
    ax.set_title(title)
    ax.set_xlabel('Bin')
plt.tight_layout()
plt.show()
print('Số chiều:', vector.size, 'L2:', np.linalg.norm(vector))

## 3. Trích train/dev và lưu cache

Mỗi ảnh được đọc một lần trong vòng trích feature. Các hàng của ma trận giữ đúng thứ tự manifest; feature và nhãn được lưu để các bước sau dùng lại.


In [ ]:
X_train = extract_matrix(train_manifest, DATA_ROOT, 'HSV12')
X_dev = extract_matrix(dev_manifest, DATA_ROOT, 'HSV12')
y_train = train_manifest.label_vnd.to_numpy(dtype=np.int64)
y_dev = dev_manifest.label_vnd.to_numpy(dtype=np.int64)
save_features(OUTPUT, train_manifest, dev_manifest, X_train, X_dev)
print(X_train.shape, X_dev.shape, X_train.dtype)

## 4. Khảo sát SVC trên dev

Linear kiểm tra ranh giới tuyến tính trong không gian histogram; RBF cho ranh giới phi tuyến. `C` điều khiển mức phạt lỗi, còn `gamma` điều khiển độ cục bộ của RBF. Mọi mô hình chỉ fit trên train.


In [ ]:
grid, best_pred, best_params = run_dev_grid(X_train, y_train, X_dev, y_dev)
scores = save_dev_result(OUTPUT, FEATURE_ID, grid, best_pred, dev_manifest)
save_json(OUTPUT / 'run_config.json', {
    'feature_id': FEATURE_ID, 'feature_dim': int(X_train.shape[1]),
    'preprocessing': 'RGB uint8 224x224 OpenCV INTER_LINEAR',
    'feature_pipeline': FEATURE_PIPELINE,
    'svm_grid': 'linear C=(0.1,1,10); rbf C=(1,10), gamma=(scale,10)',
    'class_weight': None, 'cache_size_mb': 512,
    'decision_function_shape': 'ovr', 'break_ties': True,
    'best_params': best_params, 'fit_split': 'train', 'selection_split': 'dev',
})
print('Cấu hình thắng trên dev:')
display(grid.head(7))
print('Accuracy:', scores['accuracy'], 'Macro-F1:', scores['macro_f1'],
      'MAE VND/ảnh:', scores['mae_vnd'])
display(pd.DataFrame(scores['per_class'])[
    ['denomination_vnd', 'support', 'precision', 'recall', 'f1']])

## 5. Ma trận nhầm lẫn

Hàng biểu diễn nhãn thật, cột biểu diễn nhãn SVM dự đoán. Bảng từng lớp và CSV dự đoán được lưu cùng grid để kiểm tra lại kết quả.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(np.asarray(scores['confusion_matrix']),
                       display_labels=CLASSES).plot(ax=ax, cmap='Blues',
                                                     values_format='d', xticks_rotation=45)
ax.set_title(f'{FEATURE_ID} + SVM — dev')
plt.tight_layout()
plt.savefig(OUTPUT / 'confusion_matrix_dev.png', dpi=180)
plt.savefig(OUTPUT / 'confusion_matrix_dev.pdf', dpi=180)
plt.show()